# Этап 1 — Анализ данных

Датасет: **Diabetes 130-US hospitals for years 1999–2008** (UCI 296),
101 766 строк, 50 признаков.

In [ ]:
# Подключаем стандартные библиотеки для работы с данными
import sys
sys.path.insert(0, "../src")  # Добавляем папку src в путь поиска, чтобы импортировать utils.py

import numpy as np           # Числовые вычисления (массивы, статистика)
import pandas as pd          # Таблицы и операции над ними (аналог Excel в Python)
import matplotlib.pyplot as plt  # Рисование графиков
import seaborn as sns        # Красивые графики поверх matplotlib

import utils  # Наши вспомогательные функции (загрузка, сплит, метрики)

SEED = 42  # Фиксируем случайность — чтобы результаты воспроизводились
np.random.seed(SEED)

pd.set_option("display.max_columns", 60)         # Показывать до 60 колонок в выводе
pd.set_option("display.float_format", "{:.3f}".format)  # Числа с 3 знаками после запятой
sns.set_theme(style="whitegrid")  # Стиль фона у графиков

print("OK")

## 1. Загрузка данных

In [ ]:
# Загружаем CSV-файл в DataFrame. 
# utils.load_data внутри передаёт na_values="?", то есть символ "?" в данных заменяется на NaN (пустое значение).
df = utils.load_data("../data/raw/diabetic_data.csv")

# Смотрим размер датасета: сколько строк и колонок
print("shape:", df.shape)

# Смотрим, какие типы данных в колонках (числовые, текстовые и т.д.)
print("\ndtypes:")
print(df.dtypes.value_counts())

# Выводим первые 3 строки, чтобы увидеть как выглядят данные
df.head(3)

## 2. Целевая переменная и баланс классов

В датасете readmitted (повторная госпитализация) хранится в трех вариантах:
- `NO`: нет повторной госпитализации
- `>30`: повторная госпитализация позже 30 дней
- `<30`: повторная госпитализация раньше 30 дней

Для задачи бинарной классификации считаем позитивным классом только `<30`:<br>
1 - пациент вернулся в больницу меньше чем через 30 дней, 0 - остальные случаи.

В итоге получается дисбаланс классов: позитивный класс (<30): 11.2%

In [ ]:
print("=== Три класса повторной госпитализации ===")
print(df["readmitted"].value_counts())
print()

# Создаём бинарную целевую переменную.
# Нас интересует РАННЯЯ реадмиссия — пациент вернулся в больницу менее чем через 30 дней.
# Именно "<30" — это медицинский критерий: реадмиссия за 30 дней — показатель низкого качества выписки.
# ">30" и "NO" объединяем в 0 (не ранняя реадмиссия).
df["target"] = (df["readmitted"] == "<30").astype(int)

pos_rate = df["target"].mean()
print(f"Позитивный класс (<30): {pos_rate:.1%} ({df['target'].sum()} из {len(df)})")
# ~11% позитивных — значит классы несбалансированы. Это важно для выбора метрики и способа обучения.

# Визуализируем распределение классов
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Левый график — все три исходных значения readmitted
df["readmitted"].value_counts().plot.bar(ax=axes[0], color="steelblue")
axes[0].set_title("Повторная госпитализация (3 класса)")
axes[0].set_xlabel("")

# Правый график — бинарная версия (0 и 1)
df["target"].value_counts().plot.bar(ax=axes[1], color=["steelblue", "tomato"])
axes[1].set_title(f"target (бинарный)")
axes[1].set_xlabel("")

plt.tight_layout()
plt.show()

## 3. Пропуски (NaN)

Проверяем пропущенные значения в исходном датасете. Такая проверка нужна, чтобы заранее определить проблемные признаки.<br>
Колонки с очень большим числом пропусков лучше удалить, а признаки с умеренным количеством пропусков можно обработать.<br>


Здесь же видно, что в датасете есть колонки, где более 80% ячееек пустые, такие колонки в дальнейнем учитываться не будут.

In [ ]:
# Считаем количество пропусков (NaN) в каждой колонке.
# Оставляем только те, у которых хоть один пропуск есть.
# Сортируем по убыванию, чтобы увидеть самые проблемные колонки первыми.
missing = (
    df.isnull().sum()
    .pipe(lambda s: s[s > 0])         # Фильтруем: только колонки с пропусками
    .sort_values(ascending=False)      # Сортируем: больше пропусков — выше
    .to_frame("count")
)
# Добавляем колонку с процентом пропусков от общего числа строк
missing["percent"] = missing["count"] / len(df) * 100
print(missing.to_string())

print("\nПроблемные колонки (>5% пропусков):")
# Колонки с более чем 5% пропусков — кандидаты на удаление или специальную обработку
problem_cols = missing[missing["percent"] > 5].index.tolist()
print(problem_cols)

## 4. Числовые признаки

Смотрим все признаки, которые pandas определил как числовые. Сюда попадают как настоящие признаки, так и нет.<br>

Что реально можно учитывать как числовые признаки: `time_in_hospital`, `num_lab_procedures`,
`num_procedures`, `num_medications`, `number_outpatient`, `number_emergency`, `number_inpatient`,
`number_diagnoses`. Эти признаки описывают длительность госпитализации, количество процедур, лекарств, визитов и диагнозов.

Что не стоит учитывать как обычные числа: `encounter_id` и `patient_nbr`, потому что это идентификаторы записей и пациентов и они не несут медицинского смысла. И также: `admission_type_id`, `discharge_disposition_id` и `admission_source_id`. Они записаны числами, но являются категориями из справочника `IDS_mapping.csv`.

In [ ]:
# Pandas автоматически определяет числовые колонки (int, float).
# Среди них есть как настоящие измеримые признаки (количество дней, процедур),
# так и ID-коды (encounter_id, admission_type_id), которые числами НЕ являются по смыслу.
num_cols = df.select_dtypes(include="number").columns.drop(["target"], errors="ignore")
print("Числовые признаки:", num_cols.tolist())
# Вывод позволяет вручную отделить настоящие числа от кодов — это нужно для следующего этапа.

In [ ]:
# Список только настоящих числовых признаков (без ID-кодов)
plot_cols = [
    "time_in_hospital", "num_lab_procedures", "num_procedures",
    "num_medications", "number_outpatient", "number_emergency",
    "number_inpatient", "number_diagnoses",
]
plot_cols = [c for c in plot_cols if c in df.columns]

# Рисуем гистограммы — показывают, как распределены значения каждого признака.
# Ищем выбросы, смещения, редкие значения — всё это влияет на качество модели.
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.flat, plot_cols):
    df[col].hist(ax=ax, bins=30, color="steelblue", edgecolor="white")
    ax.set_title(col, fontsize=9)
plt.suptitle("Числовые признаки — распределения", y=1.02)
plt.tight_layout()
plt.show()

## 5. Категориальные признаки

Смотрим категориальные признаки с их кардинальностью, то есть количество уникальных значений в каждой колонке.

Признаки с небольшой кардинальностью, например `gender`, `race`, `change`, `diabetesMed` и признаки лекарств, можно обрабатывать кодированием. Они имеют ограниченное число состояний.

Отдельно выделяются высококардинальные признаки: `diag_1`, `diag_2`, `diag_3` и `medical_specialty`. Их нельзя просто кодировать как отдельные категории, потому что получится слишком много редких признаков.

In [ ]:
# Берём все категориальные (текстовые) колонки, кроме readmitted (это целевая переменная)
cat_cols = df.select_dtypes(include=["object", "string", "str"]).columns.drop(["readmitted"], errors="ignore")

# Кардинальность — количество уникальных значений в колонке.
# Маленькая кардинальность (2-10): можно кодировать напрямую (one-hot, label encoding).
# Большая кардинальность (сотни): нужна группировка или embedding, иначе модель получает 
# тысячи лишних признаков и "захлёбывается".
cardinality = df[cat_cols].nunique().sort_values(ascending=False)
print(cardinality.to_string())

# Признаки с >20 уникальными значениями — требуют особой обработки
high_card = cardinality[cardinality > 20].index.tolist()
print("\nВысококардинальные (>20 уникальных):", high_card)

## 6. Диагнозы (diag_1/2/3) - распределение кодов ICD-9

Отдельно рассмотрим признаки `diag_1`, `diag_2` и `diag_3`, которые содержат ICD-9 коды диагнозов. Эти колонки важны, потому что диагнозы напрямую описывают состояние пациента и могут влиять на риск повторной госпитализации.

При этом использовать исходные коды как обычные категории неудобно: у `diag_1`, `diag_2` и `diag_3` сотни уникальных значений, многие из них
встречаются редко. Поэтому мы смотрим количество уникальных значений и самые частые ICD-9 коды. Это подтверждает, что
диагнозы нужно не оставлять в исходном виде, а укрупнить в медицинские группы.

In [ ]:
# Смотрим, сколько уникальных ICD-9 кодов в каждом столбце диагнозов.
# ICD-9 — международная классификация болезней версии 9 (числовые коды вида 250.1, 428, E895).
# 700+ уникальных значений — это слишком много для прямого кодирования.
# Решение: сгруппировать похожие коды в ~9 клинических категорий (Diabetes, Circulatory и т.д.)
for col in ["diag_1", "diag_2", "diag_3"]:
    if col in df.columns:
        print(f"{col}: {df[col].nunique()} уникальных значений")
        print(df[col].value_counts().head(10))  # Самые частые коды
        print()

## 7. discharge_disposition_id - коды смерти/хосписа

Особенно важны коды `{11, 13, 14, 19, 20, 21}`. В справочнике они соответствуют смерти пациента или выписке в хоспис: `Expired`, `Hospice / home`, `Hospice / medical facility` и похожие варианты. Такие коды учитываться не будут, так как такие пациенты повторно госпитализированы быть не могут.

Остальные значения `discharge_disposition_id` можно учитывать как категориальный признак, потому что тип выписки может быть связан с риском повторной госпитализации.

In [ ]:
# Коды типа выписки. Нас интересуют специфические коды — смерть и хоспис.
# Пациенты с этими кодами НЕ могут быть реадмитированы в принципе (они умерли или в хосписе).
# Их метка readmitted="NO" правильная, но по другой причине, чем у живых — это шум в данных.
EXCLUDE_IDS = {11, 13, 14, 19, 20, 21}  # 11=умер, 13/14=хоспис, 19/20/21=другие смертельные

ddi = df["discharge_disposition_id"].value_counts().sort_index()
print("Топ значений discharge_disposition_id:")
print(ddi.to_string())

n_exclude = df["discharge_disposition_id"].isin(EXCLUDE_IDS).sum()
print(f"\nСтрок с кодами {{11,13,14,19,20,21}} (умершие/хоспис): {n_exclude} ({n_exclude/len(df):.1%})")
# Примерно 2-3% строк будут удалены на этапе препроцессинга

## 8. Корреляции числовых признаков

Корреляция показывает, насколько линейно связаны признаки между собой и с фактом повторной госпитализации меньше чем за 30 дней.

По корреляционной матрице видно, что сильной линейной связи между отдельными числовыми признаками и `target` нет. Самая заметная корреляция у `number_inpatient` - 0.17. Остальные имеют слабую положительную связь и даже слабую отрицательную.

По итогу, по одной корреляции признаки удалять нельзя. Все связи потенциально могут полезны в сочетании с друг с другом.

In [ ]:
# Добавляем целевую переменную в список для корреляции
corr_cols = [c for c in plot_cols if c in df.columns] + ["target"]

# Матрица корреляций Пирсона: значения от -1 до +1.
# +1 — признаки растут вместе, -1 — растут в противоположных направлениях, 0 — нет линейной связи.
# Корреляция с target показывает, насколько линейно признак связан с реадмиссией.
corr = df[corr_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, linewidths=0.5)
plt.title("Корреляции числовых признаков")
plt.tight_layout()
plt.show()
# Вывод: слабые корреляции с target (max ~0.17) — это НЕ значит, что признаки бесполезны.
# Нелинейные модели (CatBoost, TabM) умеют улавливать нелинейные зависимости.

## 9. Дубликаты пациентов

Проверяем, сколько в датасете повторных визитов одних и тех же пациентов. Для этого сравниваем общее число строк с количеством уникальных `patient_nbr`. Результат показывает, что в данных 101 766 строк, но только 71 518 уникальных пациентов.

Это важно для обучения модели. Если разные визиты одного пациента попадут одновременно в train и test, модель может частично узнать пациента по похожим признакам.

Поэтому на этапе предобработки нужно оставить только первый визит каждого пациента, а затем удалить `patient_nbr` и `encounter_id`.

In [ ]:
total_rows = len(df)
unique_patients = df["patient_nbr"].nunique()  # Уникальных пациентов по ID
duplicate_visits = total_rows - unique_patients  # Повторные визиты = строки без уникального пациента

print(f"Всего строк:          {total_rows:,}")
print(f"Уникальных пациентов: {unique_patients:,}")
print(f"Повторных визитов:    {duplicate_visits:,} ({duplicate_visits/total_rows:.1%})")
# ~30% строк — повторные визиты одного пациента.
# Проблема: если разные визиты одного пациента попадут в train и test одновременно —
# модель "видела" этого пациента при обучении и "узнаёт" его на тесте. 
# Это data leakage (утечка данных) — результаты на тесте будут завышены.
# Решение в препроцессинге: оставляем только ПЕРВЫЙ визит каждого пациента.

## Итоговое резюме

Целевая переменная `readmitted` была приведена к бинарному виду: `target = 1`, если пациент повторно госпитализирован меньше чем за 30 дней. Позитивный класс составляет около 11.2%, поэтому при обучении нужно учитывать дисбаланс.

В данных есть признаки с большим количеством пропусков: `weight`, `max_glu_serum`, `A1Cresult`, `medical_specialty`, `payer_code`. Часть из них нужно удалить, часть обработать через отдельные категории.

Числовые признаки нужно разделять по смыслу. Настоящие количественные признаки можно использовать как числа, а ID-коды и идентификаторы нельзя интерпретировать как обычные числовые значения.

Категориальные признаки `diag_1`, `diag_2`, `diag_3` имеют слишком много уникальных ICD-9 кодов, поэтому дальше их нужно сгруппировать в укрупненные медицинские категории.

Строки с `discharge_disposition_id` ∈ `{11, 13, 14, 19, 20, 21}` нужно удалить, потому что они соответствуют смерти или хоспису.

Также обнаружено много повторных визитов пациентов: около 29.7% строк. Чтобы избежать утечки данных, дальше нужно оставить только первый визит каждого пациента и удалить идентификаторы `encounter_id`, `patient_nbr`.